# Day 3 — Gold Layer Practice Questions
## Topic: PySpark aggregations · RFM scoring · ntile · SQL RANK / LAG / anti-join

---

> **Rules:**
> - Silver tables must exist (run Day 2 first)
> - Q1 and Q2 use PySpark; Q3 uses SQL via SQLAlchemy
> - Write your answer before checking the hint

| # | Difficulty | Topic |
|---|-----------|-------|
| Q1 | 🟢 Easy | PySpark: weekly revenue rollup |
| Q2 | 🟡 Medium | PySpark: RFM scoring with ntile window |
| Q3 | 🔴 Hard | SQL: Top product per category using RANK() PARTITION BY |

---
## Setup — run once

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from config.db_config import (
    get_engine, set_spark_env, get_spark, new_batch,
    spark_read_pg, spark_write_pg, JDBC_URL, DB_USER, DB_PASS
)
from sqlalchemy import text

engine = get_engine()
_, GOLD_AT = new_batch()

set_spark_env()
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType

spark = get_spark('Day3_PQ')

def read_silver(table):
    return spark_read_pg(spark, 'silver', table)

print('Setup ready.')

---
---
## Q1 — 🟢 Easy
### PySpark: weekly revenue rollup

---

### Problem Statement

From `silver.orders` (excluding cancelled) using PySpark:

1. Read `silver.orders` via `read_silver('orders')` → PySpark DataFrame
2. Filter out cancelled orders: `.filter(F.col('is_cancelled') == False)`
3. Cast `order_date` to `DateType()` and `total_amount` to `DoubleType()`
4. Add `week_start` using `F.date_trunc('week', F.col('order_date'))`
5. GroupBy `week_start`, compute:
   - `total_revenue` = `F.round(F.sum('total_amount'), 2)`
   - `order_count`   = `F.count('order_id')`
6. Add `avg_order_value = F.round(F.col('total_revenue') / F.col('order_count'), 2)`
7. Sort by `week_start` ascending
8. Show top 5 rows, write to `gold.pq_weekly_sales` using `spark_write_pg(df, 'gold', 'pq_weekly_sales')`

**Hints:** `F.date_trunc('week', F.col('order_date'))`, `.agg(F.round(F.sum(...),2).alias('total_revenue'), ...)`, `.orderBy('week_start')`

In [ ]:
# Q1 Setup
df_ord = read_silver('orders')
print(f'silver.orders: {df_ord.count()} rows')
df_ord.select('order_id','order_date','total_amount','is_cancelled').show(3)

In [ ]:
# Q1 — Write your solution here


<details><summary>💡 Solution</summary>

```python
from pyspark.sql.types import DateType

df_active = (df_ord
    .filter(F.col('is_cancelled') == False)
    .withColumn('order_date',   F.col('order_date').cast(DateType()))
    .withColumn('total_amount', F.col('total_amount').cast(DoubleType()))
    .withColumn('week_start',   F.date_trunc('week', F.col('order_date')))
)

weekly = (df_active
    .groupBy('week_start')
    .agg(
        F.round(F.sum('total_amount'), 2).alias('total_revenue'),
        F.count('order_id').alias('order_count'),
    )
    .withColumn('avg_order_value', F.round(F.col('total_revenue') / F.col('order_count'), 2))
    .orderBy('week_start')
)

print('Weekly sales (first 5):')
weekly.show(5)

spark_write_pg(
    weekly.withColumn('_gold_loaded_at', F.lit(GOLD_AT)),
    'gold', 'pq_weekly_sales'
)
```
</details>

---
---
## Q2 — 🟡 Medium
### PySpark: RFM scoring with ntile window

---

### Problem Statement

Using PySpark, build a simplified RFM (Recency, Frequency, Monetary) score for each customer:

1. Read `silver.orders` (non-cancelled), compute per customer:
   - `recency_days`: `F.datediff(F.current_date(), F.max('order_date'))`
   - `frequency`: `F.count('order_id')`
   - `monetary`: `F.round(F.sum('total_amount'), 2)`
2. Score each metric from 1–4 using `F.ntile(4).over(window)`:
   - `r_score`: ntile over `recency_days` ascending, then invert: `(5 - ntile_val)` so low recency = high score
   - `f_score`: ntile over `frequency` ascending
   - `m_score`: ntile over `monetary` ascending
3. Compute `rfm_score = r_score + f_score + m_score`
4. Assign `segment` using `F.when`:
   - `rfm_score >= 10` → `'Champion'`
   - `rfm_score >= 7`  → `'Loyal'`
   - `rfm_score >= 4`  → `'At Risk'`
   - otherwise          → `'Lost'`
5. Show the segment distribution using `.groupBy('segment').count().show()`
6. Write to `gold.pq_rfm`

**Hints:** `F.ntile(4).over(Window.orderBy('col'))`, inverting R: `(F.lit(5) - F.col('r_raw'))`, `.cast(IntegerType())`

In [ ]:
# Q2 Setup
df_ord = read_silver('orders')
df_active = (df_ord
    .filter(F.col('is_cancelled') == False)
    .withColumn('total_amount', F.col('total_amount').cast(DoubleType()))
    .withColumn('order_date', F.to_date(F.col('order_date')))
)
print(f'Non-cancelled orders: {df_active.count()}')

In [ ]:
# Q2 — Write your solution here


<details><summary>💡 Solution</summary>

```python
# Step 1: aggregate per customer
rfm_raw = (df_active
    .groupBy('customer_id')
    .agg(
        F.datediff(F.current_date(), F.max('order_date').cast('date')).alias('recency_days'),
        F.count('order_id').alias('frequency'),
        F.round(F.sum('total_amount'), 2).alias('monetary'),
    )
)

# Step 2: ntile scoring
w_r = Window.orderBy('recency_days')
w_f = Window.orderBy('frequency')
w_m = Window.orderBy('monetary')

rfm_scored = (rfm_raw
    .withColumn('r_raw',   F.ntile(4).over(w_r))
    .withColumn('r_score', (F.lit(5) - F.col('r_raw')).cast(IntegerType()))
    .withColumn('f_score', F.ntile(4).over(w_f).cast(IntegerType()))
    .withColumn('m_score', F.ntile(4).over(w_m).cast(IntegerType()))
    .withColumn('rfm_score', F.col('r_score') + F.col('f_score') + F.col('m_score'))
)

# Step 3: segment assignment
rfm_final = rfm_scored.withColumn('segment',
    F.when(F.col('rfm_score') >= 10, F.lit('Champion'))
     .when(F.col('rfm_score') >= 7,  F.lit('Loyal'))
     .when(F.col('rfm_score') >= 4,  F.lit('At Risk'))
     .otherwise(F.lit('Lost'))
)

print('Segment distribution:')
rfm_final.groupBy('segment').count().show()

spark_write_pg(
    rfm_final.withColumn('_gold_loaded_at', F.lit(GOLD_AT)),
    'gold', 'pq_rfm'
)
print(f'gold.pq_rfm: {rfm_final.count()} rows')
```
</details>

---
---
## Q3 — 🔴 Hard
### SQL: Top product per category by revenue using RANK() PARTITION BY

---

### Problem Statement

Write a single SQL query (using CTEs) that returns the **#1 ranked product by revenue within each category**.  
Use `RANK() OVER (PARTITION BY category ORDER BY revenue DESC)`.

Return: `category`, `product_id`, `product_name`, `total_revenue`, `rank_in_category`  
Filter to show only rank = 1 rows.  
Sort by `total_revenue` descending.

### Expected Output (5 rows — one per category)
```
category     | product_id | product_name        | total_revenue | rank_in_category
ELECTRONICS  | P015       | Smart Watch         | ...           | 1
FOOTWEAR     | P002       | Running Shoes       | ...           | 1
...
```

**Hint:** Use a CTE — compute revenue per product first, then apply RANK in the outer query.

In [ ]:
# Q3 — Write your SQL here, then read the result via PySpark JDBC
sql = """
-- Your SQL here (put it in a view named gold.v_pq_top_product_per_cat, then read below)
"""

# Create the view with your SQL, then read it:
# with engine.connect() as conn:
#     conn.execute(text('CREATE OR REPLACE VIEW gold.v_pq_top_product_per_cat AS ' + sql))
#     conn.commit()
# spark_read_pg(spark, 'gold', 'v_pq_top_product_per_cat').show(truncate=False)

<details><summary>💡 Solution</summary>

```python
inner_sql = """
    WITH product_revenue AS (
        SELECT
            p.category,
            oi.product_id,
            p.product_name,
            ROUND(SUM(oi.line_total)::NUMERIC, 2) AS total_revenue
        FROM silver.order_items oi
        JOIN silver.products p ON oi.product_id = p.product_id
        GROUP BY p.category, oi.product_id, p.product_name
    ),
    ranked AS (
        SELECT *,
            RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS rank_in_category
        FROM product_revenue
    )
    SELECT category, product_id, product_name, total_revenue, rank_in_category
    FROM ranked
    WHERE rank_in_category = 1
    ORDER BY total_revenue DESC
"""

with engine.connect() as conn:
    conn.execute(text(
        f'CREATE OR REPLACE VIEW gold.v_pq_top_product_per_cat AS {inner_sql}'
    ))
    conn.commit()

spark_read_pg(spark, 'gold', 'v_pq_top_product_per_cat').show(truncate=False)
```
</details>

In [ ]:
spark.stop()
print('Spark stopped.')